In [ ]:
# TRINETRA: revised reproducible training/evaluation pipeline
# Intended for Kaggle. Run this file as one notebook cell or as a Python script.
# The notebook keeps the DSDRec-style filtering and leave-last-out split described
# in the manuscript, while making the implementation choices explicit in outputs.

import os
import gc
import json
import math
import random
import time
import warnings
import subprocess
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Kaggle often has PyTorch but not both modeling libraries. Install only if missing;
# this does not upgrade or replace the existing PyTorch installation.
try:
    from transformers import GPT2Config, GPT2Model
except ImportError:
    print("Installing missing dependency: transformers")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    from transformers import GPT2Config, GPT2Model

try:
    from torch_geometric.nn import GATv2Conv
except ImportError:
    print("Installing missing dependency: torch-geometric")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    from torch_geometric.nn import GATv2Conv

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors


# =============================================================================
# 1. Configuration — verify these values against the configuration that generated
#    the manuscript's reported numbers before using any result in the paper.
# =============================================================================
DATASET_NAME = "NYC"  # Set to "TKY" for Tokyo.
CSV_PATH = "/kaggle/input/foursquare/foursquare_nyc.csv"
# For TKY, edit CSV_PATH to the mounted Tokyo CSV path in your Kaggle dataset.
OUTPUT_DIR = "/kaggle/working/trinetra_outputs"

# For a quick smoke test, temporarily set RUN_SEEDS = [42]. For final journal
# results, use multiple independent seeds; five are configured here by default.
RUN_SEEDS = [42, 43, 44, 45, 46]
PREPROCESS_SEED = 42  # Keeps clustering/preprocessing fixed across training seeds.

MIN_CHECKINS_PER_USER = 10
POI_MIN_VISITS = 10
MAX_SEQ_LEN = 100

# These settings match the current manuscript's implementation-details section.
EMB_DIM = 128
DROPOUT = 0.35
CROSS_ATTN_HEADS = 8
GPT2_LAYERS = 2
GPT2_HEADS = 4
CAT_EMB_DIM = 32
CLUSTER_EMB_DIM = 16
N_CLUSTERS = 50
SPATIAL_K_BY_DATASET = {"NYC": 21, "TKY": 31}
SPATIAL_K = SPATIAL_K_BY_DATASET.get(DATASET_NAME.upper(), 21)

BATCH_SIZE = 128
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.2
LAMBDA_CL = 0.5
LAMBDA_LOC = 0.5
CL_TEMPERATURE = 0.15

# The manuscript does not specify the number of graph-pretraining epochs; 10 is
# retained from the supplied implementation and is recorded in the config output.
PHASE1_EPOCHS = 10
PHASE1_LEARNING_RATE = 1e-3
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

ACC_KS = (1, 5, 10, 20)
MRR_KS = (5, 20)  # These are the MRR cutoffs reported in the manuscript table.

# To keep graph-sequence training tractable on Kaggle, graph embeddings are
# computed once per epoch; gradients from all sequence minibatches are accumulated
# at the graph-embedding tensor and propagated through the GNN once per epoch.
# This remains joint training, but the graph encoder is updated once per epoch,
# not once per minibatch. Report this implementation detail in any revised paper.
GRAPH_ENCODER_UPDATE_FREQUENCY = "once_per_epoch_gradient_accumulation"


# =============================================================================
# 2. Reproducibility and utility functions
# =============================================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Full bitwise determinism is not guaranteed for all PyG CUDA operations.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def resolve_csv_path(path: str, dataset_name: str) -> str:
    candidate = Path(path)
    if candidate.exists():
        return str(candidate)
    # Helpful recovery for Kaggle dataset mount names that differ from the default.
    root = Path("/kaggle/input")
    token = "nyc" if dataset_name.upper() == "NYC" else "tky"
    if root.exists():
        matches = [p for p in root.rglob("*.csv") if token in p.name.lower()]
        if len(matches) == 1:
            print(f"CSV_PATH not found; using the only {dataset_name} candidate: {matches[0]}")
            return str(matches[0])
        if len(matches) > 1:
            candidates = "\n".join(str(p) for p in matches[:20])
            raise FileNotFoundError(
                f"CSV_PATH={path!r} does not exist and multiple {dataset_name} CSVs were found. "
                f"Set CSV_PATH explicitly. Candidates:\n{candidates}"
            )
    raise FileNotFoundError(
        f"Could not find CSV_PATH={path!r}. Attach the Foursquare dataset to Kaggle "
        "and set CSV_PATH to its exact mounted path."
    )


def haversine_meters(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in meters; inputs are degrees."""
    earth_radius_m = 6_371_000.0
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    a = np.clip(a, 0.0, 1.0)  # Protect against floating-point roundoff.
    return earth_radius_m * (2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a)))


def normalize_column_name(name: str) -> str:
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())


def standardize_columns(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Resolve common Foursquare CSV column variants without silently guessing values."""
    df = raw_df.copy()
    lookup = {normalize_column_name(c): c for c in df.columns}
    aliases = {
        "user_id": ["userid", "user_id", "uid", "user"],
        "venue_id": ["venueid", "venue_id", "poiid", "poi_id", "locationid", "location_id", "placeid"],
        "timestamp": ["utctimestamp", "timestamp", "time", "datetime", "checkintime"],
        "latitude": ["latitude", "lat"],
        "longitude": ["longitude", "lon", "lng", "long"],
        "category_raw": ["venuecategoryid", "venue_category_id", "categoryid", "category_id", "category", "venuecategory", "venue_category"],
    }
    missing = []
    for target, variants in aliases.items():
        source = next((lookup[normalize_column_name(v)] for v in variants
                       if normalize_column_name(v) in lookup), None)
        if source is not None:
            df[target] = df[source]
        elif target == "category_raw":
            df[target] = "__UNKNOWN_CATEGORY__"
            warnings.warn(
                "No venue category column was found. A single UNKNOWN category is used; "
                "this differs from experiments using category metadata.", RuntimeWarning
            )
        else:
            missing.append(target)
    if missing:
        raise ValueError(
            f"Missing required columns {missing}. CSV columns found: {list(raw_df.columns)}"
        )
    return df


def load_and_preprocess(csv_path: str):
    print(f"Loading data: {csv_path}")
    raw = pd.read_csv(csv_path)
    df = standardize_columns(raw)

    rows_before_cleaning = len(df)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["user_id", "venue_id", "timestamp", "latitude", "longitude"]).copy()
    valid_geo = df["latitude"].between(-90, 90) & df["longitude"].between(-180, 180)
    df = df.loc[valid_geo].copy()
    df["category_raw"] = df["category_raw"].fillna("__UNKNOWN_CATEGORY__").astype(str)
    print(f"Rows removed for missing/invalid required values: {rows_before_cleaning - len(df):,}")

    # Preserve the filtering order in the supplied implementation: POIs first,
    # then users. The source paper states the thresholds but does not specify an
    # iterative k-core procedure, so this code does not silently add one.
    venue_counts = df["venue_id"].value_counts()
    valid_venues = venue_counts.index[venue_counts >= POI_MIN_VISITS]
    df = df[df["venue_id"].isin(valid_venues)].copy()

    user_counts = df["user_id"].value_counts()
    valid_users = user_counts.index[user_counts >= MIN_CHECKINS_PER_USER]
    df = df[df["user_id"].isin(valid_users)].copy()

    if df.empty:
        raise ValueError("No records remain after cleaning and filtering. Check CSV_PATH and column mapping.")

    # Encode IDs after filtering; 0 is reserved exclusively for padding.
    user_codes, user_uniques = pd.factorize(df["user_id"], sort=True)
    venue_codes, venue_uniques = pd.factorize(df["venue_id"], sort=True)
    cat_codes, cat_uniques = pd.factorize(df["category_raw"], sort=True)
    df["user_enc"] = user_codes.astype(np.int64)
    df["venue_enc"] = venue_codes.astype(np.int64) + 1
    df["cat_enc"] = cat_codes.astype(np.int64) + 1

    n_users = int(df["user_enc"].nunique())
    n_venues = int(df["venue_enc"].max()) + 1
    n_categories = int(df["cat_enc"].max()) + 1

    # Coordinates are catalogue-level/static POI features, as assumed by the
    # transductive known-catalogue setting. Coordinates are averaged by POI.
    venue_coords = df.groupby("venue_enc", sort=True)[["latitude", "longitude"]].mean()
    lat_mean = float(venue_coords["latitude"].mean())
    lat_std = float(venue_coords["latitude"].std(ddof=1))
    lon_mean = float(venue_coords["longitude"].mean())
    lon_std = float(venue_coords["longitude"].std(ddof=1))
    if not np.isfinite(lat_std) or lat_std <= 0:
        lat_std = 1.0
    if not np.isfinite(lon_std) or lon_std <= 0:
        lon_std = 1.0
    venue_coords["lat_norm"] = (venue_coords["latitude"] - lat_mean) / lat_std
    venue_coords["lon_norm"] = (venue_coords["longitude"] - lon_mean) / lon_std

    final_venue_counts = df["venue_id"].value_counts()
    final_user_counts = df["user_id"].value_counts()
    print("\nFiltered dataset summary")
    print(f"  Rows/check-ins: {len(df):,}")
    print(f"  Users:          {n_users:,} (minimum={final_user_counts.min()})")
    print(f"  POIs incl. PAD: {n_venues:,}; real POIs={n_venues - 1:,} (post-user-filter min visits={final_venue_counts.min()})")
    print(f"  Categories:     {n_categories - 1:,} real categories")
    print("  Reminder: report these observed counts; do not copy table counts if your CSV variant differs.")

    metadata = {
        "input_rows": int(rows_before_cleaning),
        "clean_rows": int(len(df)),
        "n_users": n_users,
        "n_venues_including_padding": n_venues,
        "n_real_venues": n_venues - 1,
        "n_real_categories": n_categories - 1,
        "min_user_checkins_after_filtering": int(final_user_counts.min()),
        "min_venue_visits_after_filtering": int(final_venue_counts.min()),
        "category_source_values": int(df["category_raw"].nunique()),
        "filtering": "one pass: remove POIs with <10 visits, then users with <10 retained check-ins",
        "coordinates_and_category_metadata": "catalogue-level features; transductive known-POI setting",
    }
    return df, venue_coords, (lat_mean, lat_std, lon_mean, lon_std), n_users, n_venues, n_categories, metadata


def create_sample(venues, times, lats, lons, uid, target_idx, coord_stats):
    """Create one next-POI example using history strictly before the target."""
    start_idx = max(0, target_idx - MAX_SEQ_LEN)
    seq_v = venues[start_idx:target_idx]
    seq_t = times[start_idx:target_idx]
    seq_lat = lats[start_idx:target_idx]
    seq_lon = lons[start_idx:target_idx]
    if len(seq_v) == 0:
        raise ValueError("An empty history was passed to create_sample.")

    delta_times = [0.0]
    delta_dists = [0.0]
    for j in range(1, len(seq_v)):
        dt_seconds = max(0.0, (seq_t[j] - seq_t[j - 1]).total_seconds())
        distance_m = float(haversine_meters(seq_lat[j - 1], seq_lon[j - 1], seq_lat[j], seq_lon[j]))
        delta_times.append(dt_seconds)
        delta_dists.append(max(0.0, distance_m))

    temporal = [(int(t.hour), int(t.dayofweek)) for t in seq_t]
    pad_len = MAX_SEQ_LEN - len(seq_v)
    seq_v = [0] * pad_len + [int(v) for v in seq_v]
    temporal = [(0, 0)] * pad_len + temporal
    delta_times = [0.0] * pad_len + [math.log1p(x) for x in delta_times]
    delta_dists = [0.0] * pad_len + [math.log1p(x) for x in delta_dists]

    lat_mean, lat_std, lon_mean, lon_std = coord_stats
    target_lat = float(lats[target_idx])
    target_lon = float(lons[target_idx])
    target_coord = ((target_lat - lat_mean) / lat_std, (target_lon - lon_mean) / lon_std)
    target_venue = int(venues[target_idx])
    if target_venue <= 0:
        raise ValueError("Target venue ID must be positive; zero is reserved for padding.")
    return (seq_v, temporal, delta_times, delta_dists, int(uid), target_venue, target_coord)


def create_splits(df: pd.DataFrame, coord_stats):
    """DSDRec-style split: train all targets up to n-2, validate n-1, test n (1-indexed)."""
    train_samples, val_samples, test_samples = [], [], []
    train_prefixes = []
    grouped = df.sort_values(["user_enc", "timestamp"], kind="mergesort").groupby("user_enc", sort=True)

    for uid, group in tqdm(grouped, desc="Building chronological samples"):
        group = group.sort_values("timestamp", kind="mergesort")
        venues = group["venue_enc"].astype(int).tolist()
        times = group["timestamp"].tolist()
        lats = group["latitude"].astype(float).tolist()
        lons = group["longitude"].astype(float).tolist()
        n = len(venues)
        if n < 3:
            continue

        # Last = test, penultimate = validation, earlier targets = train.
        test_samples.append(create_sample(venues, times, lats, lons, uid, n - 1, coord_stats))
        val_samples.append(create_sample(venues, times, lats, lons, uid, n - 2, coord_stats))
        for target_idx in range(1, n - 2):  # includes index n-3, excludes validation/test targets
            train_samples.append(create_sample(venues, times, lats, lons, uid, target_idx, coord_stats))

        # Transition graph is built from the true chronological training prefix,
        # including the final training target's incoming transition. It excludes
        # validation and test check-ins/transitions.
        train_prefixes.append(venues[:n - 2])

    if not train_samples or not val_samples or not test_samples:
        raise ValueError("Empty train/validation/test split. Check filtering and trajectory lengths.")

    print("\nSplit summary")
    print(f"  Train samples: {len(train_samples):,}")
    print(f"  Validation samples: {len(val_samples):,}")
    print(f"  Test samples: {len(test_samples):,}")
    print(f"  Validation/test samples should each equal the retained user count: {len(val_samples):,}/{len(test_samples):,}")
    return train_samples, val_samples, test_samples, train_prefixes


# =============================================================================
# 3. Graph and catalogue feature construction
# =============================================================================
def build_graphs(df, venue_coords, n_venues, n_categories, train_prefixes, spatial_k):
    print("\nBuilding graph structures and POI features...")
    num_real_venues = n_venues - 1
    if num_real_venues < 2:
        raise ValueError("At least two real POIs are required to build the geographic graph.")
    if spatial_k >= num_real_venues:
        spatial_k = num_real_venues - 1
        warnings.warn(f"SPATIAL_K reduced to {spatial_k} because of the number of POIs.")

    coords_full = np.zeros((n_venues, 2), dtype=np.float64)
    for venue_id, row in venue_coords.iterrows():
        coords_full[int(venue_id)] = [float(row["latitude"]), float(row["longitude"])]
    valid_coords = coords_full[1:]

    # K-means clusters are based on known catalogue coordinates. Fixed preprocessing
    # seed ensures cluster IDs remain comparable between training seeds.
    actual_clusters = min(N_CLUSTERS, num_real_venues)
    kmeans = KMeans(n_clusters=actual_clusters, random_state=PREPROCESS_SEED, n_init=10)
    cluster_ids = kmeans.fit_predict(valid_coords)
    venue_clusters_full = np.zeros(n_venues, dtype=np.int64)
    venue_clusters_full[1:] = cluster_ids + 1

    # Use the modal category consistently for per-POI category lookup and one-hot node features.
    cat_mode = df.groupby("venue_enc")["cat_enc"].agg(lambda values: int(values.mode().iloc[0])).to_dict()
    venue_cat_ids = np.zeros(n_venues, dtype=np.int64)
    for vid, cat in cat_mode.items():
        venue_cat_ids[int(vid)] = int(cat)

    latlon_norm = np.zeros((n_venues, 2), dtype=np.float32)
    for vid, row in venue_coords.iterrows():
        latlon_norm[int(vid), 0] = float(row["lat_norm"])
        latlon_norm[int(vid), 1] = float(row["lon_norm"])
    venue_cat_onehot = np.zeros((n_venues, n_categories), dtype=np.float32)
    for vid in range(1, n_venues):
        venue_cat_onehot[vid, venue_cat_ids[vid]] = 1.0
    graph_feats = np.concatenate([venue_cat_onehot, latlon_norm], axis=1).astype(np.float32)

    # Transition graph from training prefixes only. Preserve the supplied design's
    # bidirectional edges; edge multiplicity is collapsed to graph topology.
    transition_pairs = set()
    for prefix in train_prefixes:
        for a, b in zip(prefix[:-1], prefix[1:]):
            if a == b:
                continue
            transition_pairs.add((int(a), int(b)))
            transition_pairs.add((int(b), int(a)))
    if not transition_pairs:
        raise ValueError("No transition edges were built from training prefixes.")
    trans_array = np.asarray(sorted(transition_pairs), dtype=np.int64)
    edge_index_trans = torch.as_tensor(trans_array.T.copy(), dtype=torch.long)

    # True great-circle nearest neighbours: sklearn's haversine metric expects
    # radians in [latitude, longitude] order, not degrees or raw Euclidean lat/lon.
    coords_rad = np.radians(valid_coords)
    knn = NearestNeighbors(n_neighbors=min(spatial_k + 1, len(coords_rad)),
                           algorithm="ball_tree", metric="haversine")
    knn.fit(coords_rad)
    _, neighbor_rows = knn.kneighbors(coords_rad)
    geo_pairs = set()
    for i, row in enumerate(neighbor_rows):
        neighbors = [int(j) for j in row if int(j) != i][:spatial_k]
        src = i + 1
        for j in neighbors:
            dst = j + 1
            if src != dst:
                geo_pairs.add((src, dst))
                geo_pairs.add((dst, src))
    if not geo_pairs:
        raise ValueError("No geographic edges were built.")
    geo_array = np.asarray(sorted(geo_pairs), dtype=np.int64)
    edge_index_geo = torch.as_tensor(geo_array.T.copy(), dtype=torch.long)

    print(f"  Transition directed edges after symmetrization: {edge_index_trans.shape[1]:,}")
    print(f"  Geographic directed edges after symmetrization: {edge_index_geo.shape[1]:,}")
    print(f"  Haversine K used: {spatial_k}; KMeans clusters used: {actual_clusters}")
    return (
        torch.from_numpy(graph_feats), edge_index_trans, edge_index_geo,
        torch.from_numpy(venue_cat_ids), torch.from_numpy(venue_clusters_full),
        trans_array, int(spatial_k), int(actual_clusters)
    )


# =============================================================================
# 4. Dataset and model
# =============================================================================
class SequenceDataset(Dataset):
    """Tensorize each split once to avoid recreating hundreds of tensors per epoch."""
    def __init__(self, samples):
        self.seq = torch.from_numpy(np.asarray([s[0] for s in samples], dtype=np.int64))
        self.time_pair = torch.from_numpy(np.asarray([s[1] for s in samples], dtype=np.int64))
        self.delta_time = torch.from_numpy(np.asarray([s[2] for s in samples], dtype=np.float32))
        self.delta_dist = torch.from_numpy(np.asarray([s[3] for s in samples], dtype=np.float32))
        self.user_ids = torch.from_numpy(np.asarray([s[4] for s in samples], dtype=np.int64))
        self.targets = torch.from_numpy(np.asarray([s[5] for s in samples], dtype=np.int64))
        self.coordinates = torch.from_numpy(np.asarray([s[6] for s in samples], dtype=np.float32))

    def __len__(self):
        return int(self.targets.shape[0])

    def __getitem__(self, index):
        return (self.seq[index], self.time_pair[index], self.delta_time[index],
                self.delta_dist[index], self.user_ids[index], self.targets[index],
                self.coordinates[index])


class IntervalEmbedding(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.linear = nn.Linear(2, emb_dim)
        self.activation = nn.Tanh()

    def forward(self, delta_time, delta_dist):
        values = torch.stack([delta_time, delta_dist], dim=-1)
        return self.activation(self.linear(values))


class TimeEmbedding(nn.Module):
    def __init__(self, emb_dim, hour_dim=16, dow_dim=8):
        super().__init__()
        self.hour_embedding = nn.Embedding(24, hour_dim)
        self.day_embedding = nn.Embedding(7, dow_dim)
        self.projection = nn.Linear(hour_dim + dow_dim, emb_dim)

    def forward(self, time_pair):
        hour = self.hour_embedding(time_pair[:, :, 0])
        day = self.day_embedding(time_pair[:, :, 1])
        return self.projection(torch.cat([hour, day], dim=-1))


class DualGraphEncoderGAT(nn.Module):
    def __init__(self, input_dim, hidden_dim=EMB_DIM, dropout=DROPOUT):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        self.transition_gat = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, dropout=dropout)
        self.geographic_gat = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.fusion_gate = nn.Linear(hidden_dim * 2, 1)

    def forward(self, features, edge_index_transition, edge_index_geo):
        x = F.elu(self.input_projection(features))
        h_transition = self.transition_gat(x, edge_index_transition)
        h_geo = self.geographic_gat(x, edge_index_geo)
        fusion = torch.cat([h_transition, h_geo], dim=-1)
        alpha = torch.sigmoid(self.fusion_gate(fusion))
        return self.layer_norm(self.dropout(alpha * h_transition + (1.0 - alpha) * h_geo))


class VenuePredictorGPT2(nn.Module):
    def __init__(self, num_venues, num_categories, num_clusters, emb_dim=EMB_DIM, dropout=DROPOUT):
        super().__init__()
        self.venue_embedding = nn.Embedding(num_venues, emb_dim, padding_idx=0)
        self.category_embedding = nn.Embedding(num_categories, CAT_EMB_DIM, padding_idx=0)
        self.cluster_embedding = nn.Embedding(num_clusters, CLUSTER_EMB_DIM, padding_idx=0)
        self.time_embedding = TimeEmbedding(emb_dim)
        self.interval_embedding = IntervalEmbedding(emb_dim)
        self.input_projection = nn.Linear(emb_dim + CAT_EMB_DIM + CLUSTER_EMB_DIM, emb_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=emb_dim, num_heads=CROSS_ATTN_HEADS,
            batch_first=True, dropout=dropout
        )
        self.cross_layer_norm = nn.LayerNorm(emb_dim)

        config = GPT2Config(
            vocab_size=1,  # inputs_embeds are supplied directly; GPT-2 token lookup is unused.
            n_embd=emb_dim,
            n_layer=GPT2_LAYERS,
            n_head=GPT2_HEADS,
            n_inner=emb_dim * 4,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
            n_positions=MAX_SEQ_LEN,
            n_ctx=MAX_SEQ_LEN,
            use_cache=False,
        )
        self.transformer = GPT2Model(config)
        self.classifier = nn.Linear(emb_dim, num_venues)
        self.coordinate_regressor = nn.Linear(emb_dim, 2)

    def forward(self, seq_ids, time_pair, delta_time, delta_dist, graph_view, category_lookup, cluster_lookup):
        batch_size, seq_len = seq_ids.shape
        category_ids = category_lookup[seq_ids]
        cluster_ids = cluster_lookup[seq_ids]

        venue_emb = self.venue_embedding(seq_ids)
        category_emb = self.category_embedding(category_ids)
        cluster_emb = self.cluster_embedding(cluster_ids)
        base = self.input_projection(torch.cat([venue_emb, category_emb, cluster_emb], dim=-1))
        seq_rep = base + self.time_embedding(time_pair) + self.interval_embedding(delta_time, delta_dist)

        graph_keys = graph_view[seq_ids]
        attn_out, _ = self.cross_attention(
            query=seq_rep,
            key=graph_keys,
            value=graph_keys,
            key_padding_mask=(seq_ids == 0),
            need_weights=False,
        )
        fused = self.cross_layer_norm(seq_rep + attn_out)
        attention_mask = (seq_ids != 0).long()
        transformer_output = self.transformer(inputs_embeds=fused, attention_mask=attention_mask)

        # Histories are left-padded, so the latest real POI is always the final position.
        last_hidden = transformer_output.last_hidden_state[:, -1, :]
        logits = self.classifier(last_hidden)
        pred_coordinates = self.coordinate_regressor(last_hidden)
        return logits, pred_coordinates, last_hidden


class TRINETRA(nn.Module):
    def __init__(self, encoder, predictor, graph_features, edge_index_trans, edge_index_geo,
                 venue_categories, venue_clusters):
        super().__init__()
        self.encoder = encoder
        self.predictor = predictor
        self.register_buffer("graph_features", graph_features)
        self.register_buffer("edge_index_trans", edge_index_trans)
        self.register_buffer("edge_index_geo", edge_index_geo)
        self.register_buffer("venue_categories", venue_categories)
        self.register_buffer("venue_clusters", venue_clusters)

    def encode_graph(self):
        return self.encoder(self.graph_features, self.edge_index_trans, self.edge_index_geo)

    def predict_with_graph(self, seq, time_pair, delta_time, delta_dist, graph_view):
        return self.predictor(
            seq, time_pair, delta_time, delta_dist, graph_view,
            self.venue_categories, self.venue_clusters
        )


class ContrastiveLoss(nn.Module):
    """In-batch sequence-to-target-POI contrastive objective used in the supplied code."""
    def __init__(self, temperature=CL_TEMPERATURE):
        super().__init__()
        self.temperature = temperature
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, sequence_representation, target_representation):
        similarity = torch.matmul(sequence_representation, target_representation.transpose(0, 1))
        similarity = similarity / self.temperature
        diagonal_targets = torch.arange(similarity.size(0), device=similarity.device)
        return self.cross_entropy(similarity, diagonal_targets)


def classification_loss_without_padding(logits, targets, label_smoothing=LABEL_SMOOTHING):
    """Cross entropy over real POIs only; smoothing mass is not assigned to PAD class 0."""
    if torch.any(targets <= 0):
        raise ValueError("Training/evaluation targets must be valid POI IDs > 0.")
    real_poi_logits = logits[:, 1:]
    shifted_targets = targets - 1
    log_probs = F.log_softmax(real_poi_logits, dim=-1)
    nll = -log_probs.gather(1, shifted_targets.unsqueeze(1)).squeeze(1)
    smooth_loss = -log_probs.mean(dim=-1)
    return ((1.0 - label_smoothing) * nll + label_smoothing * smooth_loss).mean()


def logits_for_ranking(logits):
    # Padding remains excluded from every reported ranking metric.
    return logits.masked_fill(torch.arange(logits.size(1), device=logits.device).unsqueeze(0) == 0,
                              torch.finfo(logits.dtype).min)


def batch_metrics(logits, targets):
    ranked_logits = logits_for_ranking(logits)
    max_k = min(max(max(ACC_KS), max(MRR_KS)), ranked_logits.size(1) - 1)
    top_indices = torch.topk(ranked_logits, k=max_k, dim=1).indices
    hits = top_indices.eq(targets.unsqueeze(1))
    acc_values, mrr_values = [], []
    for k in ACC_KS:
        if k <= max_k:
            acc_values.append(hits[:, :k].any(dim=1).float().mean().item())
        else:
            acc_values.append(float("nan"))
    for k in MRR_KS:
        if k <= max_k:
            hits_k = hits[:, :k]
            found = hits_k.any(dim=1).float()
            rank = hits_k.float().argmax(dim=1).float() + 1.0
            mrr_values.append(((1.0 / rank) * found).mean().item())
        else:
            mrr_values.append(float("nan"))
    return acc_values, mrr_values


def make_loader(dataset, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    kwargs = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return DataLoader(**kwargs)


def negative_edges(num_venues, positive_edges, count, rng):
    """Sample unique directed non-edges; no padding node, self-loop, or known positive edge."""
    positive_np = positive_edges.detach().cpu().numpy().T
    positive_keys = np.unique(positive_np[:, 0].astype(np.int64) * num_venues + positive_np[:, 1])
    selected_keys = set()
    attempts = 0
    while len(selected_keys) < count:
        attempts += 1
        if attempts > 1000:
            raise RuntimeError("Negative edge sampler could not find enough non-edges.")
        remaining = count - len(selected_keys)
        sample_count = max(1024, remaining * 2)
        src = rng.integers(1, num_venues, size=sample_count, dtype=np.int64)
        dst = rng.integers(1, num_venues, size=sample_count, dtype=np.int64)
        keys = src * num_venues + dst
        valid = (src != dst) & (~np.isin(keys, positive_keys))
        candidates = np.unique(keys[valid])
        for key in candidates:
            selected_keys.add(int(key))
            if len(selected_keys) >= count:
                break
    keys = np.fromiter(selected_keys, dtype=np.int64, count=count)
    src = keys // num_venues
    dst = keys % num_venues
    return torch.as_tensor(np.stack([src, dst]), dtype=torch.long, device=positive_edges.device)


def run_graph_pretraining(model, phase1_epochs, seed):
    print(f"\nPhase 1: graph link-prediction pretraining ({phase1_epochs} epochs)")
    graph_optimizer = torch.optim.Adam(model.encoder.parameters(), lr=PHASE1_LEARNING_RATE)
    bce = nn.BCEWithLogitsLoss()
    rng = np.random.default_rng(seed + 10_000)
    positive_edges = model.edge_index_trans
    if positive_edges.shape[1] == 0:
        raise ValueError("Transition graph contains no positive edges for link prediction.")

    for epoch in range(1, phase1_epochs + 1):
        model.encoder.train()
        graph_optimizer.zero_grad(set_to_none=True)
        node_embeddings = model.encode_graph()
        src_pos, dst_pos = positive_edges[0], positive_edges[1]
        pos_scores = (node_embeddings[src_pos] * node_embeddings[dst_pos]).sum(dim=-1)
        neg = negative_edges(model.graph_features.size(0), positive_edges, src_pos.numel(), rng)
        src_neg, dst_neg = neg[0], neg[1]
        neg_scores = (node_embeddings[src_neg] * node_embeddings[dst_neg]).sum(dim=-1)
        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        loss = bce(scores, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), GRAD_CLIP_NORM)
        graph_optimizer.step()
        print(f"  Phase 1 epoch {epoch:02d}/{phase1_epochs} | link BCE={loss.item():.5f}")

    # Copy pretrained real-POI graph representations into item embeddings. PAD row 0
    # intentionally remains zero; it is not a real POI and is never transferred.
    model.encoder.eval()
    model.predictor.eval()
    with torch.no_grad():
        node_embeddings = model.encode_graph()
        model.predictor.venue_embedding.weight[1:].copy_(node_embeddings[1:])
    print("Phase 1 complete; graph representations copied to real-POI embeddings.")


def evaluate(model, loader, split_name):
    model.eval()
    total_cls_loss = 0.0
    sample_count = 0
    acc_sum = np.zeros(len(ACC_KS), dtype=np.float64)
    mrr_sum = np.zeros(len(MRR_KS), dtype=np.float64)
    model.encoder.eval()
    with torch.no_grad():
        # The global graphs are static for this evaluation pass; encode once rather than
        # recomputing GNN embeddings for every mini-batch.
        graph_view = model.encode_graph()
        for batch in loader:
            seq, time_pair, dt, dd, uid, target, coord_target = [x.to(DEVICE, non_blocking=True) for x in batch]
            logits, pred_coords, sequence_rep = model.predict_with_graph(seq, time_pair, dt, dd, graph_view)
            loss = classification_loss_without_padding(logits, target)
            accs, mrrs = batch_metrics(logits, target)
            batch_size = target.size(0)
            total_cls_loss += loss.item() * batch_size
            acc_sum += np.asarray(accs) * batch_size
            mrr_sum += np.asarray(mrrs) * batch_size
            sample_count += batch_size

    if sample_count == 0:
        raise ValueError(f"{split_name} loader is empty.")
    accs = acc_sum / sample_count
    mrrs = mrr_sum / sample_count
    macc = float(np.mean(accs))
    result = {**{f"Acc@{k}": float(v) for k, v in zip(ACC_KS, accs)},
              **{f"MRR@{k}": float(v) for k, v in zip(MRR_KS, mrrs)},
              "mAcc": macc,
              "classification_loss": total_cls_loss / sample_count,
              "n_samples": int(sample_count)}
    return result


def print_metric_result(split_name, result, elapsed=None):
    timing = f" | {elapsed:.1f}s" if elapsed is not None else ""
    print(f"[{split_name}]{timing} loss={result['classification_loss']:.5f} | mAcc={result['mAcc']:.5f}")
    print("  " + " | ".join(f"Acc@{k}={result[f'Acc@{k}']:.4f}" for k in ACC_KS))
    print("  " + " | ".join(f"MRR@{k}={result[f'MRR@{k}']:.4f}" for k in MRR_KS))


def save_model_checkpoint(path, model, epoch, validation_result, seed):
    checkpoint = {
        "encoder": {k: v.detach().cpu().clone() for k, v in model.encoder.state_dict().items()},
        "predictor": {k: v.detach().cpu().clone() for k, v in model.predictor.state_dict().items()},
        "epoch": int(epoch),
        "seed": int(seed),
        "validation_mAcc": float(validation_result["mAcc"]),
        "config": {
            "dataset": DATASET_NAME,
            "spatial_k": SPATIAL_K,
            "emb_dim": EMB_DIM,
            "dropout": DROPOUT,
            "lambda_cl": LAMBDA_CL,
            "lambda_loc": LAMBDA_LOC,
            "cl_temperature": CL_TEMPERATURE,
            "label_smoothing": LABEL_SMOOTHING,
        },
    }
    torch.save(checkpoint, path)


def load_model_checkpoint(path, model):
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # Compatibility with older PyTorch versions.
        checkpoint = torch.load(path, map_location="cpu")
    model.encoder.load_state_dict(checkpoint["encoder"])
    model.predictor.load_state_dict(checkpoint["predictor"])
    return checkpoint


def train_one_seed(seed, train_loader, val_loader, test_loader, graph_data,
                   n_venues, n_categories, actual_clusters, output_dir):
    set_seed(seed)
    print("\n" + "=" * 78)
    print(f"TRAINING TRINETRA | dataset={DATASET_NAME} | seed={seed} | device={DEVICE}")
    print("=" * 78)

    (graph_features, edge_index_trans, edge_index_geo, venue_categories,
     venue_clusters, _trans_array, _effective_k, _actual_clusters) = graph_data
    encoder = DualGraphEncoderGAT(input_dim=graph_features.shape[1], hidden_dim=EMB_DIM, dropout=DROPOUT)
    predictor = VenuePredictorGPT2(
        num_venues=n_venues,
        num_categories=n_categories,
        num_clusters=actual_clusters + 1,
        emb_dim=EMB_DIM,
        dropout=DROPOUT,
    )
    model = TRINETRA(encoder, predictor, graph_features, edge_index_trans, edge_index_geo,
                     venue_categories, venue_clusters).to(DEVICE)

    run_graph_pretraining(model, PHASE1_EPOCHS, seed)

    # Separate optimizers permit efficient sequence-gradient accumulation through the
    # epoch's global graph embedding tensor, then one GNN update per epoch.
    optimizer_predictor = torch.optim.AdamW(model.predictor.parameters(), lr=LEARNING_RATE,
                                            weight_decay=WEIGHT_DECAY)
    optimizer_encoder = torch.optim.AdamW(model.encoder.parameters(), lr=LEARNING_RATE,
                                          weight_decay=WEIGHT_DECAY)
    scheduler_predictor = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_predictor, mode="max", factor=0.5, patience=2
    )
    scheduler_encoder = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_encoder, mode="max", factor=0.5, patience=2
    )
    contrastive_loss = ContrastiveLoss(CL_TEMPERATURE)

    checkpoint_path = os.path.join(output_dir, f"best_trinetra_{DATASET_NAME}_seed{seed}.pt")
    best_validation_macc = -float("inf")
    best_epoch = 0
    patience = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        model.encoder.train()
        model.predictor.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.time()

        optimizer_encoder.zero_grad(set_to_none=True)
        graph_view = model.encode_graph()
        # Detached leaf accumulates d(loss)/d(graph node embedding) for every batch;
        # one backward through the graph encoder is done after all batches.
        graph_view_for_sequence = graph_view.detach().requires_grad_(True)

        total_objective = 0.0
        seen = 0
        train_acc_sum = np.zeros(len(ACC_KS), dtype=np.float64)
        train_mrr_sum = np.zeros(len(MRR_KS), dtype=np.float64)

        progress = tqdm(train_loader, desc=f"seed {seed} epoch {epoch}/{MAX_EPOCHS}", leave=False,
                        mininterval=10.0)
        for batch in progress:
            seq, time_pair, dt, dd, uid, target, coord_target = [x.to(DEVICE, non_blocking=True) for x in batch]
            optimizer_predictor.zero_grad(set_to_none=True)
            logits, pred_coords, sequence_rep = model.predict_with_graph(
                seq, time_pair, dt, dd, graph_view_for_sequence
            )
            target_rep = model.predictor.venue_embedding(target)
            cls_loss = classification_loss_without_padding(logits, target)
            cl_loss = contrastive_loss(sequence_rep, target_rep)
            loc_loss = F.mse_loss(pred_coords, coord_target)
            objective = cls_loss + LAMBDA_CL * cl_loss + LAMBDA_LOC * loc_loss
            objective.backward()
            torch.nn.utils.clip_grad_norm_(model.predictor.parameters(), GRAD_CLIP_NORM)
            optimizer_predictor.step()

            batch_size = target.size(0)
            total_objective += float(objective.detach().item()) * batch_size
            accs, mrrs = batch_metrics(logits.detach(), target)
            train_acc_sum += np.asarray(accs) * batch_size
            train_mrr_sum += np.asarray(mrrs) * batch_size
            seen += batch_size

        if graph_view_for_sequence.grad is None:
            raise RuntimeError("No sequence gradient reached graph embeddings; graph encoder was not updated.")
        graph_view.backward(graph_view_for_sequence.grad)
        torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), GRAD_CLIP_NORM)
        optimizer_encoder.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - start_time
        train_result = {
            "objective_loss": total_objective / max(1, seen),
            "mAcc": float(np.mean(train_acc_sum / max(1, seen))),
            "n_samples": int(seen),
        }
        train_result.update({f"Acc@{k}": float(v) for k, v in zip(ACC_KS, train_acc_sum / max(1, seen))})
        train_result.update({f"MRR@{k}": float(v) for k, v in zip(MRR_KS, train_mrr_sum / max(1, seen))})

        val_result = evaluate(model, val_loader, "VALID")
        scheduler_predictor.step(val_result["mAcc"])
        scheduler_encoder.step(val_result["mAcc"])
        print(f"[TRAIN] epoch={epoch:03d} objective={train_result['objective_loss']:.5f} "
              f"mAcc={train_result['mAcc']:.5f} | time={elapsed:.1f}s")
        print_metric_result("VALID", val_result)

        history_row = {"seed": seed, "epoch": epoch, "train_objective_loss": train_result["objective_loss"],
                       "train_mAcc": train_result["mAcc"], "epoch_seconds": elapsed,
                       **{f"val_{k}": v for k, v in val_result.items()}}
        history_rows.append(history_row)

        # Test metrics are deliberately NOT computed inside the epoch loop.
        if val_result["mAcc"] > best_validation_macc:
            best_validation_macc = val_result["mAcc"]
            best_epoch = epoch
            patience = 0
            save_model_checkpoint(checkpoint_path, model, epoch, val_result, seed)
            print(f"  Saved best validation checkpoint (epoch {epoch}, val mAcc={best_validation_macc:.5f}).")
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping after {epoch} epochs; best epoch was {best_epoch}.")
                break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pd.DataFrame(history_rows).to_csv(
        os.path.join(output_dir, f"history_{DATASET_NAME}_seed{seed}.csv"), index=False
    )
    checkpoint = load_model_checkpoint(checkpoint_path, model)
    print(f"\nEvaluating the test split ONCE using selected epoch {checkpoint['epoch']}.")
    test_result = evaluate(model, test_loader, "TEST")
    print_metric_result("TEST", test_result)

    row = {"dataset": DATASET_NAME, "seed": seed, "best_epoch": int(checkpoint["epoch"]),
           "best_validation_mAcc": float(checkpoint["validation_mAcc"]),
           **{f"test_{k}": v for k, v in test_result.items()}}
    return row


# =============================================================================
# 5. Main execution
# =============================================================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = resolve_csv_path(CSV_PATH, DATASET_NAME)
DATASET_NAME = DATASET_NAME.upper()
if DATASET_NAME not in ("NYC", "TKY"):
    raise ValueError("DATASET_NAME must be 'NYC' or 'TKY'.")
SPATIAL_K = SPATIAL_K_BY_DATASET[DATASET_NAME]

config_dump = {
    "dataset_name": DATASET_NAME,
    "csv_path": CSV_PATH,
    "run_seeds": RUN_SEEDS,
    "preprocess_seed": PREPROCESS_SEED,
    "min_checkins_per_user": MIN_CHECKINS_PER_USER,
    "poi_min_visits": POI_MIN_VISITS,
    "max_seq_len": MAX_SEQ_LEN,
    "embedding_dimension": EMB_DIM,
    "dropout": DROPOUT,
    "cross_attention_heads": CROSS_ATTN_HEADS,
    "gpt2_layers": GPT2_LAYERS,
    "gpt2_heads": GPT2_HEADS,
    "n_clusters": N_CLUSTERS,
    "spatial_k": SPATIAL_K,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "lambda_cl": LAMBDA_CL,
    "lambda_loc": LAMBDA_LOC,
    "contrastive_temperature": CL_TEMPERATURE,
    "phase1_epochs": PHASE1_EPOCHS,
    "phase1_learning_rate": PHASE1_LEARNING_RATE,
    "graph_encoder_update_frequency": GRAPH_ENCODER_UPDATE_FREQUENCY,
    "acc_cutoffs": ACC_KS,
    "mrr_cutoffs": MRR_KS,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
}
with open(os.path.join(OUTPUT_DIR, f"config_{DATASET_NAME}.json"), "w", encoding="utf-8") as file:
    json.dump(config_dump, file, indent=2)
print("Configuration saved to:", os.path.join(OUTPUT_DIR, f"config_{DATASET_NAME}.json"))
print("Device:", DEVICE)
if len(RUN_SEEDS) < 5:
    print("WARNING: fewer than five training seeds are configured. Use at least five for the final robustness analysis.")

(df, venue_coords, coord_stats, n_users, n_venues, n_categories, data_metadata) = load_and_preprocess(CSV_PATH)
train_samples, val_samples, test_samples, train_prefixes = create_splits(df, coord_stats)
(graph_features, edge_index_trans, edge_index_geo, venue_categories, venue_clusters,
 transition_array, effective_k, actual_clusters) = build_graphs(
    df, venue_coords, n_venues, n_categories, train_prefixes, SPATIAL_K
)

# Tensorize once. This reduces Python-object overhead and avoids rebuilding datasets
# for every seed. The original sample lists can be released after tensorization.
train_dataset = SequenceDataset(train_samples)
val_dataset = SequenceDataset(val_samples)
test_dataset = SequenceDataset(test_samples)
train_sample_count, val_sample_count, test_sample_count = len(train_dataset), len(val_dataset), len(test_dataset)
del train_samples, val_samples, test_samples
gc.collect()

# Build loaders per seed; seed-specific generators control shuffling reproducibly.
graph_data = (graph_features, edge_index_trans, edge_index_geo, venue_categories,
              venue_clusters, transition_array, effective_k, actual_clusters)

# Save observed preprocessing summary separately from source-paper table values.
with open(os.path.join(OUTPUT_DIR, f"data_summary_{DATASET_NAME}.json"), "w", encoding="utf-8") as file:
    json.dump({**data_metadata, "train_samples": train_sample_count,
               "validation_samples": val_sample_count, "test_samples": test_sample_count,
               "spatial_k": effective_k,
               "transition_edges": int(edge_index_trans.shape[1]),
               "geographic_edges": int(edge_index_geo.shape[1])}, file, indent=2)

all_run_results = []
for seed in RUN_SEEDS:
    train_loader = make_loader(train_dataset, shuffle=True, seed=seed)
    val_loader = make_loader(val_dataset, shuffle=False, seed=seed)
    test_loader = make_loader(test_dataset, shuffle=False, seed=seed)
    result = train_one_seed(
        seed, train_loader, val_loader, test_loader, graph_data,
        n_venues=n_venues, n_categories=n_categories, actual_clusters=actual_clusters,
        output_dir=OUTPUT_DIR,
    )
    all_run_results.append(result)
    pd.DataFrame(all_run_results).to_csv(
        os.path.join(OUTPUT_DIR, f"seed_results_{DATASET_NAME}.csv"), index=False
    )

results_df = pd.DataFrame(all_run_results)
results_path = os.path.join(OUTPUT_DIR, f"seed_results_{DATASET_NAME}.csv")
results_df.to_csv(results_path, index=False)

metric_columns = [c for c in results_df.columns if c.startswith("test_") and
                  (c.endswith("mAcc") or c.startswith("test_Acc@") or c.startswith("test_MRR@"))]
summary_rows = []
for metric in metric_columns:
    values = pd.to_numeric(results_df[metric], errors="coerce")
    summary_rows.append({"metric": metric.replace("test_", ""), "mean": values.mean(),
                         "std_sample": values.std(ddof=1) if len(values) > 1 else np.nan,
                         "n_seeds": int(values.notna().sum())})
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_DIR, f"mean_std_summary_{DATASET_NAME}.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 78)
print("FINAL MULTI-SEED TEST SUMMARY")
print("=" * 78)
print(results_df.to_string(index=False))
print("\nMean and sample standard deviation across seeds:")
print(summary_df.to_string(index=False))
print(f"\nSaved seed results: {results_path}")
print(f"Saved summary:      {summary_path}")
print(f"All outputs are in: {OUTPUT_DIR}")
print("IMPORTANT: rerun and report results produced by this exact code/configuration; do not reuse old table values.")
